# VKR Anomaly Monitoring

Ноутбук запускает полный экспериментальный pipeline: предобработка исходного Kaggle dataset, построение manifest логического объединения, обучение моделей, оценка и формирование HTML-отчета.

In [ ]:
%pip install -q -r requirements.txt


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path('.')
SRC_ROOT = PROJECT_ROOT / 'src'
sys.path.append(str(PROJECT_ROOT))
sys.path.append(str(SRC_ROOT))

KAGGLE_INPUT = Path('/kaggle/input')
KAGGLE_WORKING = Path('/kaggle/working')
OUTPUT_ROOT = KAGGLE_WORKING / 'vkr_anomaly_monitoring_outputs'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print('KAGGLE_INPUT:', KAGGLE_INPUT)
print('KAGGLE_WORKING:', KAGGLE_WORKING)
print('OUTPUT_ROOT:', OUTPUT_ROOT)
print('Available input folders:')
for p in sorted(KAGGLE_INPUT.iterdir()):
    print('-', p)

In [ ]:
from src.utils.io import load_yaml
from src.utils.paths import get_config_path

CONFIG_PATH = get_config_path()
config = load_yaml(CONFIG_PATH)
execution = config.get('execution', {})
print('Config path:', CONFIG_PATH)
print('Execution config:', execution)

In [ ]:
if execution.get('run_preprocessing', True):
    from src.preprocessing.create_predictive_dataset import run_preprocessing
    preprocessing_metadata = run_preprocessing(
        config_path=CONFIG_PATH,
        batch_start=execution.get('batch_start', 0),
        batch_size=execution.get('batch_size'),
    )
    print('Processed:', preprocessing_metadata['processed_count'])
    print('Failed:', preprocessing_metadata['failed_count'])
else:
    print('Preprocessing skipped by config.')

In [ ]:
if execution.get('run_manifest_build', True):
    from src.manifest.build_logical_manifest import run_manifest_build
    manifest = run_manifest_build(config_path=CONFIG_PATH)
    print(manifest)
else:
    print('Manifest build skipped by config.')

In [ ]:
SCENARIO = execution.get('scenario', 'all')

if execution.get('run_regression_training', True):
    from src.training.train_regressor import run_training as run_regression_training
    regression_metrics = run_regression_training(CONFIG_PATH, scenario=SCENARIO)
    print('Regression metrics:', regression_metrics)
else:
    print('Regression training skipped by config.')

if execution.get('run_classification_training', True):
    from src.training.train_classifier import run_training as run_classification_training
    classification_metrics = run_classification_training(CONFIG_PATH, scenario=SCENARIO)
    print('Classification metrics:', classification_metrics)
else:
    print('Classification training skipped by config.')

In [ ]:
if execution.get('run_dashboard_report', True):
    from src.dashboard.build_monitoring_report import run_report_build
    report_path = run_report_build(CONFIG_PATH)
    print('HTML report:', report_path)
else:
    print('Dashboard report skipped by config.')